# Colab T4 Segment Sentiment Inference

런타임 → 런타임 유형 변경 → 하드웨어 가속기 GPU 선택. 권장 GPU: T4.

주의: `review_rating`과 `rating_satisfaction_signal`은 감정 정답 라벨이 아니다. 이 노트북은 segment 감정을 한국어 감정분석 모델로 예측하고, 별점 신호는 관계 분석용으로만 보존한다.

In [ ]:
!nvidia-smi
!pip -q install transformers sentencepiece accelerate pandas numpy tqdm

`segments_for_sentiment_colab.csv`를 Colab 왼쪽 파일 패널에 업로드하거나 Google Drive를 마운트한 뒤 `INPUT_PATH`를 수정하세요.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import pipeline

INPUT_PATH = Path('/content/segments_for_sentiment_colab.csv')
OUTPUT_DIR = Path('/content/colab_sentiment_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(INPUT_PATH, encoding='utf-8-sig')
df['segment_text_for_model'] = df.get('segment_text_for_model', df.get('segment_text', '')).fillna('').astype(str)
df.head()

In [ ]:
MODELS = [
    {'name': 'koelectra_nsmc_monologg', 'hf_model': 'monologg/koelectra-base-finetuned-nsmc', 'task': 'binary_sentiment'},
    {'name': 'koelectra_nsmc_documented', 'hf_model': 'cringepnh/koelectra-korean-sentiment', 'task': 'binary_sentiment'},
    {'name': 'koelectra_kote_44emotion', 'hf_model': 'tobykim/koelectra-44emotions', 'task': 'emotion_extension'},
    {'name': 'kluebert_aihub_7emotion', 'hf_model': 'dlckdfuf141/korean-emotion-kluebert-v2', 'task': 'emotion_extension'},
]

def standardize(label):
    value = str(label).upper()
    if value in {'LABEL_1', 'POSITIVE', '1', '긍정', '행복'}:
        return 'POSITIVE'
    if value in {'LABEL_0', 'NEGATIVE', '0', '부정', '분노', '슬픔', '혐오'}:
        return 'NEGATIVE'
    if 'POS' in value:
        return 'POSITIVE'
    if 'NEG' in value:
        return 'NEGATIVE'
    return 'NEUTRAL' if '중립' in str(label) or 'NEUTRAL' in value else str(label)


In [ ]:
texts = df['segment_text_for_model'].tolist()
all_rows = []
run_log = []
device = 0 if torch.cuda.is_available() else -1

for model_info in MODELS:
    name = model_info['name']
    hf_model = model_info['hf_model']
    try:
        clf = pipeline('text-classification', model=hf_model, tokenizer=hf_model, device=device, truncation=True, max_length=128)
        outputs = clf(texts, batch_size=32)
        for base, out in zip(df.to_dict('records'), outputs):
            label = out.get('label', '')
            score = float(out.get('score', 0.0))
            all_rows.append({
                'segment_id': base.get('segment_id'),
                'segment_text': base.get('segment_text'),
                'review_id': base.get('review_id'),
                'review_rating': base.get('review_rating'),
                'rating_satisfaction_signal': base.get('rating_satisfaction_signal'),
                'rating_group': base.get('rating_group'),
                'model_name': name,
                'hf_model': hf_model,
                'predicted_label_original': label,
                'predicted_label_standard': standardize(label),
                'predicted_score': score,
                'all_scores_json': json.dumps(out, ensure_ascii=False),
                'run_status': 'ok',
                'error_message': '',
            })
        run_log.append({'model_name': name, 'hf_model': hf_model, 'status': 'ok', 'rows': len(outputs), 'error_message': ''})
        del clf
        torch.cuda.empty_cache()
    except Exception as exc:
        run_log.append({'model_name': name, 'hf_model': hf_model, 'status': 'failed', 'rows': 0, 'error_message': str(exc)})
        torch.cuda.empty_cache()

pred_all = pd.DataFrame(all_rows)
pred_all.to_csv(OUTPUT_DIR / 'segment_sentiment_predictions_all_models.csv', index=False, encoding='utf-8-sig')

binary = pred_all[pred_all['model_name'].eq('koelectra_nsmc_monologg')].copy()
binary = binary.rename(columns={'predicted_label_standard': 'model_sentiment_label', 'predicted_score': 'model_sentiment_score'})
binary.to_csv(OUTPUT_DIR / 'segment_sentiment_binary_primary.csv', index=False, encoding='utf-8-sig')

emotion = pred_all[~pred_all['model_name'].eq('koelectra_nsmc_monologg')].copy()
emotion.to_csv(OUTPUT_DIR / 'segment_emotion_extension.csv', index=False, encoding='utf-8-sig')
pd.DataFrame(run_log).to_csv(OUTPUT_DIR / 'sentiment_model_run_log.csv', index=False, encoding='utf-8-sig')
pred_all.head()

In [ ]:
# 별점 신호는 정답 비교가 아니라 관계 분석용 교차표로만 확인한다.
if not binary.empty:
    relation = pd.crosstab(binary['rating_satisfaction_signal'], binary['model_sentiment_label'], normalize='index')
    relation.to_csv(OUTPUT_DIR / 'rating_signal_model_sentiment_relation.csv', encoding='utf-8-sig')
relation if 'relation' in globals() else None

다운로드할 파일: `segment_sentiment_predictions_all_models.csv`, `segment_sentiment_binary_primary.csv`, `segment_emotion_extension.csv`, `sentiment_model_run_log.csv`. 로컬의 `C:\every_time\outputs\colab_results`에 넣으세요.